# Data Preprocessing and Understanding - Bengaluru Weather Dataset


### Step 1: Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


### Step 2: Load Dataset


In [ ]:
df = pd.read_csv('Bengaluru.csv', skiprows=3)
df.head()


### Step 3: Initial Data Understanding


In [ ]:
print('Dataset Shape:', df.shape)
print('\nDataset Information:')
df.info()


In [ ]:
df.describe().T


### Step 4: Clean and Standardize Column Names


In [ ]:
import re

def clean_column_name(col):
    col = re.sub(r'\s*\([^)]*\)', '', col)
    col = col.strip().lower()
    col = re.sub(r'[^a-z0-9_]+', '_', col)
    col = re.sub(r'_+', '_', col)
    return col.strip('_')

df.columns = [clean_column_name(c) for c in df.columns]
print('Cleaned Column Names:')
print(df.columns.tolist())


### Step 5: Check and Handle Missing Values


In [ ]:
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_count, 'Percentage': missing_percentage})
print('Missing Value Summary:')
print(missing_df[missing_df['Missing_Count'] > 0])
if missing_df['Missing_Count'].sum() == 0:
    print('No missing values found in the dataset.')


### Step 6: Check and Handle Duplicate Rows


In [ ]:
duplicate_count = df.duplicated().sum()
print(f'Total Duplicate Rows: {duplicate_count}')
if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f'Shape after removing duplicates: {df.shape}')


### Step 7: Datetime Conversion and Temporal Feature Extraction


In [ ]:
df['time'] = pd.to_datetime(df['time'])
df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month
df['day'] = df['time'].dt.day
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['day_name'] = df['time'].dt.day_name()
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df['quarter'] = df['time'].dt.quarter

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8, 9]:
        return 'Monsoon'
    else:
        return 'Post_Monsoon'

df['season'] = df['month'].apply(get_season)
df[['time', 'year', 'month', 'day', 'hour', 'day_name', 'season']].head()


### Step 8: Remove Redundant and Zero-Variance Features


In [ ]:
constant_cols = [col for col in df.select_dtypes(include=[np.number]).columns if df[col].nunique() <= 1]
print('Zero-variance / Constant Columns:', constant_cols)
if constant_cols:
    df = df.drop(columns=constant_cols)
    print(f'Dropped {len(constant_cols)} constant columns.')
print('Remaining Columns Count:', df.shape[1])


### Step 9: Statistical Analysis and Correlation


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
skew_kurt_df = pd.DataFrame({
    'Skewness': df[numeric_cols].skew(),
    'Kurtosis': df[numeric_cols].kurtosis()
})
print('Skewness and Kurtosis Summary:')
print(skew_kurt_df.head(10))


In [ ]:
corr_matrix = df[numeric_cols].corr()
plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False)
plt.title('Feature Correlation Heatmap')
plt.show()


### Step 10: Outlier Detection and Capping


In [ ]:
key_features = ['temperature_2m', 'wind_speed_10m', 'precipitation', 'surface_pressure']
key_features = [f for f in key_features if f in df.columns]

outlier_summary = {}
for col in key_features:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = int(df[(df[col] < lower_bound) | (df[col] > upper_bound)][col].count())
    outlier_summary[col] = {'Q1': q1, 'Q3': q3, 'IQR': iqr, 'Lower': lower_bound, 'Upper': upper_bound, 'Outliers': outliers}

outlier_df = pd.DataFrame(outlier_summary).T
print(outlier_df)


In [ ]:
df_capped = df.copy()
for col in key_features:
    q1 = df_capped[col].quantile(0.25)
    q3 = df_capped[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)
print('Outlier capping completed.')


### Step 11: Feature Transformation and Scaling


In [ ]:
if 'precipitation' in df.columns:
    df['log_precipitation'] = np.log1p(df['precipitation'])

df_encoded = pd.get_dummies(df, columns=['season', 'day_name'], drop_first=True)
print('Shape after encoding:', df_encoded.shape)


In [ ]:
scale_features = [col for col in ['temperature_2m', 'relative_humidity_2m', 'surface_pressure', 'wind_speed_10m'] if col in df.columns]
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()

df_standardized = df.copy()
df_standardized[[f + '_standardized' for f in scale_features]] = scaler_standard.fit_transform(df[scale_features])

df_normalized = df.copy()
df_normalized[[f + '_normalized' for f in scale_features]] = scaler_minmax.fit_transform(df[scale_features])

print('Standardized Features Preview:')
print(df_standardized[[f + '_standardized' for f in scale_features]].head())


### Step 12: Save Preprocessed Dataset


In [ ]:
output_file = 'bengaluru_preprocessed.csv'
df.to_csv(output_file, index=False)
print(f'Cleaned and preprocessed data saved to {output_file}')
print(f'Final Dataset Shape: {df.shape}')
